In [20]:
import polars as pl

df = pl.DataFrame({
    "Product": ["Laptop", "Mouse", "Keyboard"],
    "Stock": [10, 50, 30]
})
print(df)

shape: (3, 2)
┌──────────┬───────┐
│ Product  ┆ Stock │
│ ---      ┆ ---   │
│ str      ┆ i64   │
╞══════════╪═══════╡
│ Laptop   ┆ 10    │
│ Mouse    ┆ 50    │
│ Keyboard ┆ 30    │
└──────────┴───────┘


In [28]:
# polars complex data manipulation example
import polars as pl

df = pl.DataFrame({
    "s.no": [1, 2, 3, 4, 8, 6, 7, 8, 9, 10],
    "Name": ["ajay", "Mark", "lena","kelvin","sora","rahul","pooja","sora","smith","laxmi"],
    "age": [25, 30, 22, 28, 24, 27, 26, 24, 31, 23],
    "gender": ["M", "M", "F", "M", "F", "M", "F", "F", "M", "F"],
    "Sports": ["Cricket", "Football", "Tennis", "Basketball", "Swimming", "Running", "Cycling", "Swimming", "Badminton", "Table Tennis"]
})

df = df.unique()

df = df.with_columns([
    pl.col("Name").str.to_uppercase(),
    pl.col("gender").str.replace("M", "Male").str.replace("F", "Female")
])

stats = df.group_by("Sports").agg([
    pl.col("age").mean().alias("Average_Age"),
    pl.len().alias("Total_Players")
])

filtered_df = df.filter((pl.col("age") > 25) & (pl.col("gender") == "Male"))

print(df)
print(stats)
print(filtered_df)

shape: (9, 5)
┌──────┬────────┬─────┬────────┬──────────────┐
│ s.no ┆ Name   ┆ age ┆ gender ┆ Sports       │
│ ---  ┆ ---    ┆ --- ┆ ---    ┆ ---          │
│ i64  ┆ str    ┆ i64 ┆ str    ┆ str          │
╞══════╪════════╪═════╪════════╪══════════════╡
│ 9    ┆ SMITH  ┆ 31  ┆ Male   ┆ Badminton    │
│ 6    ┆ RAHUL  ┆ 27  ┆ Male   ┆ Running      │
│ 4    ┆ KELVIN ┆ 28  ┆ Male   ┆ Basketball   │
│ 8    ┆ SORA   ┆ 24  ┆ Female ┆ Swimming     │
│ 10   ┆ LAXMI  ┆ 23  ┆ Female ┆ Table Tennis │
│ 3    ┆ LENA   ┆ 22  ┆ Female ┆ Tennis       │
│ 1    ┆ AJAY   ┆ 25  ┆ Male   ┆ Cricket      │
│ 7    ┆ POOJA  ┆ 26  ┆ Female ┆ Cycling      │
│ 2    ┆ MARK   ┆ 30  ┆ Male   ┆ Football     │
└──────┴────────┴─────┴────────┴──────────────┘
shape: (9, 3)
┌──────────────┬─────────────┬───────────────┐
│ Sports       ┆ Average_Age ┆ Total_Players │
│ ---          ┆ ---         ┆ ---           │
│ str          ┆ f64         ┆ u32           │
╞══════════════╪═════════════╪═══════════════╡
│ Cycling      ┆ 

In [31]:
import polars as pl

# 🔹 Step 1: Create Data
df = pl.DataFrame({
    "id": [1,2,3,4,5,6,7,8,9,10,11],
    "name": ["ajay","mark","lena","kelvin","sora","rahul","pooja","sora","smith","laxmi", None],
    "age": [25,30,22,28,24,27,26,24,31,23,None],
    "gender": ["M","M","F","M","F","M","F","F","M","F","M"],
    "salary": [20000,30000,25000,40000,None,35000,28000,27000,45000,22000,30000],
    "sports": ["Cricket","Football","Tennis","Basketball","Swimming","Running","Cycling","Swimming","Badminton","Table Tennis","Cricket"]
})

# 🔹 Step 2: Pre-calc mean age
mean_age = df.select(pl.col("age").mean()).item()

# 🔹 Step 3: Clean Data
df = df.with_columns([
    pl.col("name").fill_null("unknown").str.to_uppercase(),

    pl.col("age").fill_null(mean_age),

    pl.col("salary").fill_null(0),

    # safer gender replace
    pl.when(pl.col("gender") == "M")
      .then(pl.lit("Male"))
      .otherwise(pl.lit("Female"))
      .alias("gender")
])

# 🔹 Step 4: Remove Duplicates
df = df.unique(subset=["name","age","gender","sports"], maintain_order=True)

# 🔹 Step 5: Feature Engineering
df = df.with_columns([
    pl.when(pl.col("age") > 25)
      .then(pl.lit("Senior"))
      .otherwise(pl.lit("Junior"))
      .alias("age_group"),

    pl.when(pl.col("salary") > 30000)
      .then(pl.lit("High"))
      .otherwise(pl.lit("Low"))
      .alias("salary_level")
])

# 🔹 Step 6: Filtering
filtered = df.filter(
    (pl.col("age") > 25) &
    (pl.col("gender") == "Male")
)

# 🔹 Step 7: Sorting
sorted_df = df.sort("salary", descending=True)

# 🔹 Step 8: Group By
stats = df.group_by("sports").agg([
    pl.col("age").mean().alias("avg_age"),
    pl.col("salary").sum().alias("total_salary"),
    pl.len().alias("count")
])

# 🔹 Step 9: Join
extra = pl.DataFrame({
    "sports": ["Cricket","Football","Swimming"],
    "type": ["Outdoor","Outdoor","Water"]
})

joined = df.join(extra, on="sports", how="left")

# 🔹 Output
print(df)
print(filtered)
print(sorted_df)
print(stats)
print(joined)

shape: (10, 8)
┌─────┬─────────┬──────┬────────┬────────┬──────────────┬───────────┬──────────────┐
│ id  ┆ name    ┆ age  ┆ gender ┆ salary ┆ sports       ┆ age_group ┆ salary_level │
│ --- ┆ ---     ┆ ---  ┆ ---    ┆ ---    ┆ ---          ┆ ---       ┆ ---          │
│ i64 ┆ str     ┆ f64  ┆ str    ┆ i64    ┆ str          ┆ str       ┆ str          │
╞═════╪═════════╪══════╪════════╪════════╪══════════════╪═══════════╪══════════════╡
│ 1   ┆ AJAY    ┆ 25.0 ┆ Male   ┆ 20000  ┆ Cricket      ┆ Junior    ┆ Low          │
│ 2   ┆ MARK    ┆ 30.0 ┆ Male   ┆ 30000  ┆ Football     ┆ Senior    ┆ Low          │
│ 3   ┆ LENA    ┆ 22.0 ┆ Female ┆ 25000  ┆ Tennis       ┆ Junior    ┆ Low          │
│ 4   ┆ KELVIN  ┆ 28.0 ┆ Male   ┆ 40000  ┆ Basketball   ┆ Senior    ┆ High         │
│ 5   ┆ SORA    ┆ 24.0 ┆ Female ┆ 0      ┆ Swimming     ┆ Junior    ┆ Low          │
│ 6   ┆ RAHUL   ┆ 27.0 ┆ Male   ┆ 35000  ┆ Running      ┆ Senior    ┆ High         │
│ 7   ┆ POOJA   ┆ 26.0 ┆ Female ┆ 28000  ┆ Cycling